In [ ]:
import pandas as pd
from pathlib import Path

from algo.features   import add_indicators, add_labels
from algo.model      import load_or_train, predict_last_vec, LOOKBACK
from algo.backtester import backtest

# 1)  Raw data ────────────────────────────────────────────────────
csv_path = Path("data/HDFCBANK_3minute.csv")
df = pd.read_csv(csv_path, parse_dates=["date"], index_col="date")

# 2)  Indicators → 3‑class labels
df = add_indicators(df)
df = add_labels(df, horizon=24, thr_atr_mult=1.0, drop_flat=False)   # –1 / 0 / +1

# 3)  Train / test split (walk‑forward)
split     = "2025-07-01"
df_train  = df[df.index <  split].copy()
df_test   = df[df.index >= split].copy()

# 4)  Train or load the 3‑class XGB
model = load_or_train(df_train, retrain=True, horizon=5)

# 5)  Run the ML back‑test (EV gate inside)
trades, metrics = backtest(
    df=df_test,
    model=model,
    predict_fn=predict_last_vec,   # returns p(class = long)
    capital=350_000,
    contract_size=150,
    lookback=LOOKBACK,
    # leave these as None to inherit the shared constants
    tp_atr_mult=None,
    sl_atr_mult=None,
    max_hold_bars=None,
    debug=False,
)

print(metrics)
print(trades.head())


Prepared 76628 samples | Class counts: {0: 26503, 1: 22954, 2: 27171}
🔧  Training …


In [2]:
trades

,entry_ts,exit_ts,side,entry_price,exit_price,atr,qty,gross_pnl,fees,pnl,exit_reason,ml_prob,equity
0,2025-07-03 09:42:00,2025-07-03 09:45:00,SELL,1992.2,1994.1,2.613249,150,-285.0,154.173813,-439.173813,TRAIL_SL,0.278959,349560.826187
1,2025-07-03 12:30:00,2025-07-03 12:33:00,BUY,1999.4,1999.5,1.484214,150,15.0,154.482140,-139.482140,TRAIL_SL,0.410304,349421.344047
2,2025-07-09 13:45:00,2025-07-09 14:18:00,SELL,2012.0,2010.5,1.252743,150,225.0,155.088869,69.911131,TRAIL_SL,0.215075,349491.255178
3,2025-07-09 14:21:00,2025-07-09 14:45:00,SELL,2011.0,2008.0,1.415870,150,450.0,154.970223,295.029778,TRAIL_SL,0.150179,349786.284956
4,2025-07-09 14:51:00,2025-07-09 14:57:00,SELL,2006.7,2007.6,1.796921,150,-135.0,154.908483,-289.908483,TRAIL_SL,0.462254,349496.376472
5,2025-07-09 15:12:00,2025-07-09 15:15:00,BUY,2011.0,2011.0,1.482612,150,0.0,155.100205,-155.100205,TRAIL_SL,0.219367,349341.276267
6,2025-07-09 15:18:00,2025-07-09 15:21:00,SELL,2011.4,2011.9,1.349916,150,-75.0,155.143331,-230.143331,TRAIL_SL,0.238969,349111.132937
7,2025-07-09 15:24:00,2025-07-09 15:27:00,SELL,2011.6,2008.5,1.316432,150,465.0,154.998083,310.001917,TRAIL_SL,0.168268,349421.134854
8,2025-07-10 10:09:00,2025-07-10 10:54:00,BUY,2004.6,2004.5,1.835029,150,-15.0,154.752480,-169.752480,TRAIL_SL,0.333367,349251.382374
9,2025-07-10 13:06:00,2025-07-10 13:09:00,SELL,2011.0,2011.4,1.542474,150,-60.0,155.117536,-215.117536,TRAIL_SL,0.326077,349036.264838


In [3]:
trades['exit_reason'].value_counts()

exit_reason
TRAIL_SL    25
SL           5
Name: count, dtype: int64